# Green AI Crop Doctor — Potato Leaf Model Training

Training and evaluation notebook only. This notebook trains the potato-leaf classifier and exports TensorFlow Lite model files plus the matching labels. The application frontend/backend are maintained separately in the GitHub project; no app-serving code is included here.

Classes: `Potato___Early_blight`, `Potato___Late_blight`, and `Potato___healthy`. This is an educational screening prototype, not a confirmed diagnosis or pesticide prescription.


In [ ]:
# Kaggle normally includes TensorFlow, NumPy, pandas, and scikit-learn.
# No pip install or codecarbon dependency is required for training/export.
print("Using the packages already available in this Kaggle runtime.")


In [ ]:

import os, json, time, shutil, pathlib, random, hashlib, math
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score

SEED = 123
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
CROP_PREFIX = "Potato___"
EXPECTED_CLASSES = {
    "Potato___Early_blight",
    "Potato___Late_blight",
    "Potato___healthy",
}
WEIGHTS = "imagenet"
HEAD_EPOCHS = 12
FINE_TUNE_EPOCHS = 8
DATA_DIR = "crop_data"
MODEL_DIR = "model"

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.makedirs(MODEL_DIR, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Visible GPUs:", tf.config.list_physical_devices("GPU"))
print("Expected potato classes:", sorted(EXPECTED_CLASSES))


In [ ]:
# Attach your potato-leaf dataset with Kaggle's Add Input button before running this cell.
# This cell searches /kaggle/input and copies the three required potato classes.
input_root = pathlib.Path("/kaggle/input")
if not input_root.exists():
    raise FileNotFoundError("/kaggle/input was not found. Run this notebook on Kaggle and attach your dataset using Add Input.")

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

found = {}
for p in sorted(input_root.rglob("*")):
    if p.is_dir() and p.name in EXPECTED_CLASSES and p.name not in found:
        found[p.name] = p

missing = EXPECTED_CLASSES - set(found)
if missing:
    raise RuntimeError(
        "Dataset folders not found under /kaggle/input: "
        + ", ".join(sorted(missing))
        + ". Check that your dataset is attached and contains the exact class folder names."
    )

for name, folder in found.items():
    shutil.copytree(folder, os.path.join(DATA_DIR, name))

print("Dataset source: /kaggle/input")
print("Copied potato classes:", sorted(found))


In [ ]:

class_names = sorted(
    p.name for p in pathlib.Path(DATA_DIR).iterdir() if p.is_dir()
)
if set(class_names) != EXPECTED_CLASSES:
    raise ValueError(
        f"Expected exactly {sorted(EXPECTED_CLASSES)}, found {class_names}"
    )

for class_name in class_names:
    files = [
        p for p in pathlib.Path(DATA_DIR, class_name).iterdir()
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ]
    print(f"{class_name:30s} {len(files):5d} images")
    if not files:
        raise ValueError(f"No images found for {class_name}")

print("\\nClass order (this order is saved with the model):", class_names)


In [ ]:

# Build the manifest and remove byte-identical duplicates. This prevents exact
# duplicate files from being counted as independent examples.
paths, labels, seen_hashes = [], [], set()
duplicate_count = 0

for idx, class_name in enumerate(class_names):
    folder = pathlib.Path(DATA_DIR, class_name)
    for image_path in sorted(folder.iterdir()):
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
        if digest in seen_hashes:
            duplicate_count += 1
            continue
        seen_hashes.add(digest)
        paths.append(str(image_path))
        labels.append(idx)

paths = np.asarray(paths)
labels = np.asarray(labels, dtype=np.int64)
print("Unique images:", len(paths), "| exact duplicates skipped:", duplicate_count)

# Stratification preserves class proportions. If source images have plant/session
# IDs, use a group-aware split instead; this dataset does not expose reliable
# plant/session identifiers in the folder structure.
train_p, temp_p, train_y, temp_y = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED
)
val_p, test_p, val_y, test_y = train_test_split(
    temp_p, temp_y, test_size=0.50, stratify=temp_y, random_state=SEED
)

print(f"train={len(train_p)}  validation={len(val_p)}  test={len(test_p)}")
for idx, name in enumerate(class_names):
    print(
        f"{name:30s} train={int((train_y == idx).sum()):4d} "
        f"val={int((val_y == idx).sum()):4d} "
        f"test={int((test_y == idx).sum()):4d}"
    )

# Balanced class weights help training but do NOT replace collecting more
# genuinely varied healthy-leaf examples.
present_classes = np.unique(train_y)
cw = compute_class_weight("balanced", classes=present_classes, y=train_y)
class_weight = {int(cls): float(weight) for cls, weight in zip(present_classes, cw)}
print("Class weights:", class_weight)


In [ ]:

def load_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(
        image_bytes, channels=3, expand_animations=False
    )
    image.set_shape([None, None, 3])
    # MobileNetV3Small below includes its own [0,255] preprocessing.
    image = tf.image.resize(image, IMG_SIZE, antialias=True)
    image = tf.cast(image, tf.float32)
    return image, label

# Augmentation is applied only to training batches. Keep it moderate so disease
# signs are not distorted beyond recognition.
augment = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.08),
    layers.RandomContrast(0.08),
], name="training_augmentation")

def make_ds(image_paths, image_labels, training=False):
    ds = tf.data.Dataset.from_tensor_slices((image_paths, image_labels))
    if training:
        ds = ds.shuffle(
            buffer_size=len(image_paths), seed=SEED, reshuffle_each_iteration=True
        )
    ds = ds.map(load_image, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(BATCH_SIZE)
    if training:
        ds = ds.map(
            lambda x, y: (augment(x, training=True), y),
            num_parallel_calls=AUTOTUNE
        )
    return ds.prefetch(AUTOTUNE)

train_ds = make_ds(train_p, train_y, training=True)
val_ds = make_ds(val_p, val_y)
test_ds = make_ds(test_p, test_y)


## 2. Model: MobileNetV3-Small with transfer learning
Small pretrained backbone, new classification head. Phase 1 trains only the head; phase 2 fine-tunes the top layers (BatchNorm kept frozen).

In [ ]:

# MobileNetV3Small is retained to keep the project lightweight.
# include_preprocessing=True means model inputs must remain pixel values 0..255.
base = tf.keras.applications.MobileNetV3Small(
    input_shape=(*IMG_SIZE, 3),
    include_top=False,
    weights=WEIGHTS,
    include_preprocessing=True,
)
base.trainable = False

inputs = layers.Input(shape=(*IMG_SIZE, 3), name="leaf_rgb_0_255")
features = base(inputs, training=False)
features = layers.GlobalAveragePooling2D()(features)
features = layers.Dropout(0.30)(features)
outputs = layers.Dense(len(class_names), activation="softmax", name="class_probabilities")(features)
model = tf.keras.Model(inputs, outputs, name="green_ai_mobilenetv3small")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.summary()


## 3. Training with energy tracking

In [ ]:

def make_callbacks():
    # Create fresh callbacks for each phase so EarlyStopping state is not reused.
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, min_lr=1e-7
        ),
        tf.keras.callbacks.TerminateOnNaN(),
    ]

print("Phase 1: train the classification head")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=HEAD_EPOCHS,
    class_weight=class_weight,
    callbacks=make_callbacks(),
)

print("\nPhase 2: fine-tune the final backbone layers")
base.trainable = True
for layer in base.layers[:-25]:
    layer.trainable = False
# Batch normalization statistics are kept frozen for small datasets.
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history_ft = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINE_TUNE_EPOCHS,
    class_weight=class_weight,
    callbacks=make_callbacks(),
)
print("Training complete. Carbon-emissions tracking is omitted to avoid an extra package dependency.")


## 4. Evaluation (Keras model)
Watch the **Late blight** row: calling a sick plant healthy is the costly mistake.

In [ ]:

# Evaluate once on the held-out test split. Do not tune thresholds on this set.
y_prob = model.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(axis=1)

keras_acc = float((y_pred == test_y).mean())
balanced_acc = float(balanced_accuracy_score(test_y, y_pred))
cm = confusion_matrix(test_y, y_pred, labels=np.arange(len(class_names)))

print(f"Keras test accuracy: {keras_acc:.4f}")
print(f"Balanced accuracy:   {balanced_acc:.4f}\\n")
print(classification_report(
    test_y, y_pred, labels=np.arange(len(class_names)),
    target_names=class_names, digits=3, zero_division=0
))
print("Confusion matrix (rows=true, columns=predicted):")
print(cm)

healthy_i = class_names.index("Potato___healthy")
early_i = class_names.index("Potato___Early_blight")
late_i = class_names.index("Potato___Late_blight")

print("\\nSafety-relevant error counts:")
print("Diseased leaves predicted Healthy:",
      int((((test_y == early_i) | (test_y == late_i)) & (y_pred == healthy_i)).sum()))
print("Healthy leaves predicted as disease:",
      int(((test_y == healthy_i) & (y_pred != healthy_i)).sum()))

# Class-specific recall helps reveal weak classes hidden by overall accuracy.
for i, name in enumerate(class_names):
    n = int((test_y == i).sum())
    recall = float(cm[i, i] / n) if n else float("nan")
    print(f"{name}: recall={recall:.3f} (support={n})")


## 5. Green AI baseline
The same task with a heavy model (ResNet50), tracked the same way, so "green" is a measured comparison and not just a claim.

In [ ]:

# Optional heavier reference model. Leave False for a faster normal run.
# Enable only when you have sufficient runtime and want a measured comparison.
RUN_BASELINE = False
baseline_info = None

if RUN_BASELINE:
    from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_pre

    b_base = tf.keras.applications.ResNet50(
        input_shape=(*IMG_SIZE, 3), include_top=False, weights=WEIGHTS
    )
    b_base.trainable = False
    b_in = layers.Input(shape=(*IMG_SIZE, 3))
    x = layers.Lambda(resnet_pre)(b_in)
    x = b_base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.2)(x)
    b_out = layers.Dense(len(class_names), activation="softmax")(x)
    baseline = tf.keras.Model(b_in, b_out)
    baseline.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

    baseline.fit(
        train_ds, validation_data=val_ds, epochs=HEAD_EPOCHS,
        class_weight=class_weight, callbacks=make_callbacks()
    )

    b_pred = baseline.predict(test_ds, verbose=0).argmax(axis=1)
    baseline_info = {
        "name": "ResNet50 (head-only reference)",
        "accuracy": float((b_pred == test_y).mean()),
        "balanced_accuracy": float(balanced_accuracy_score(test_y, b_pred)),
        "params_millions": round(baseline.count_params() / 1e6, 2),
        "approx_fp32_size_mb": round(baseline.count_params() * 4 / 1e6, 1),
        "co2_kg": None,
    }
    print(baseline_info)
else:
    print("ResNet50 reference skipped to save training time and compute.")


## 6. Export light models (TFLite FP32 and INT8)

In [ ]:

# Save model and its exact output-label order together.
model.save(os.path.join(MODEL_DIR, "leaf_disease_fp32.keras"))
with open(os.path.join(MODEL_DIR, "labels.json"), "w", encoding="utf-8") as f:
    json.dump(class_names, f, indent=2)

with open(os.path.join(MODEL_DIR, "model_metadata.json"), "w", encoding="utf-8") as f:
    json.dump({
        "project": "Green AI Crop Doctor",
        "crop": "potato",
        "architecture": "MobileNetV3Small",
        "image_size": list(IMG_SIZE),
        "input_range": "0..255 RGB",
        "classes": class_names,
        "dataset": "PlantVillage potato subset",
        "note": "Controlled-image research prototype; validate on independent field images."
    }, f, indent=2)

print("Saved Keras model, labels, and metadata.")


In [ ]:

# Export a float model and a fully-integer INT8 model.
converter = tf.lite.TFLiteConverter.from_keras_model(model)
fp32_bytes = converter.convert()
fp32_path = os.path.join(MODEL_DIR, "green_leaf_model_fp32.tflite")
with open(fp32_path, "wb") as f:
    f.write(fp32_bytes)

def representative_data_gen():
    # Representative images must use the same raw 0..255 input convention.
    calibration_paths = train_p[:min(200, len(train_p))]
    for path in calibration_paths:
        image, _ = load_image(tf.constant(path), tf.constant(0))
        yield [tf.expand_dims(image, axis=0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
int8_path = os.path.join(MODEL_DIR, "green_leaf_model_int8.tflite")
with open(int8_path, "wb") as f:
    f.write(converter.convert())

print("TFLite exports created:")
print(fp32_path, os.path.getsize(fp32_path) / 1e6, "MB")
print(int8_path, os.path.getsize(int8_path) / 1e6, "MB")


In [ ]:
shutil.make_archive("model_files", "zip", MODEL_DIR)
try:
    from google.colab import files
    files.download("model_files.zip")
except ImportError:
    print("Not on Colab: find model_files.zip in the current folder.")

## 11. Before presenting the project

- [ ] Collect more varied healthy potato-leaf images; class weights alone do not fix missing visual diversity.
- [ ] Test on independent field photographs that were not used in training or validation.
- [ ] Check the confusion matrix, especially diseased-to-Healthy mistakes and Healthy-to-disease mistakes.
- [ ] Compare Keras FP32, TFLite FP32, and TFLite INT8 on the same test images.
- [ ] Verify `labels.json` is packaged with the exact model file.
- [ ] Tune confidence and top-two-margin thresholds on validation data only.
- [ ] Do not claim the model recognizes non-potato leaves or diseases outside its three classes.
- [ ] Do not present example pesticide doses or model predictions as treatment instructions.

**Research limitation:** the PlantVillage split is image-level because the selected folder structure does not provide reliable plant/session grouping metadata. A second evaluation using farm photographs is essential before making real-world accuracy claims.


## 8. Package model artifacts for the separate application

The `model/` directory contains the exported TFLite model(s), `labels.json`, metadata, and evaluation results. Download `green_leaf_model_int8.tflite` and `labels.json` for the FastAPI application. Keep the class order exactly as saved in `labels.json`. No frontend or API server is created by this notebook.


In [ ]:
# Bundle the exported model artifacts for easy download from Kaggle.
import shutil, os
model_archive = shutil.make_archive("green_ai_model_artifacts", "zip", MODEL_DIR)
print("Created:", model_archive)
print("Files:", sorted(os.listdir(MODEL_DIR)))


## 9. Before presenting the project

- Collect more varied healthy potato-leaf images; class weights alone do not fix missing visual diversity.
- Test on independent field photographs that were not used in training or validation.
- Check the confusion matrix, especially diseased-to-Healthy mistakes and Healthy-to-disease mistakes.
- Verify `labels.json` is packaged with the exact model file.
- Tune confidence and top-two-margin thresholds on validation data only.
- Do not claim the model recognizes non-potato leaves or diseases outside its three classes.

**Research limitation:** the PlantVillage split is image-level because the selected folder structure does not provide reliable plant/session grouping metadata. A second evaluation using farm photographs is essential before making real-world accuracy claims.
